# Word Embeddings and Neural Text Classification

Word2Vec trained from scratch — both Skip-Gram and CBOW, with manual forward and
backward passes on the GPU — then a comparison against pretrained FastText
vectors on AG News topic classification.

**Headline result:** Skip-Gram and CBOW reach almost the same validation loss
(3.36 vs 3.67), yet their embedding spaces are qualitatively different.
Skip-Gram learns semantic neighbourhoods; CBOW, on this corpus size, collapses
onto function words. Loss curves alone would have hidden that completely.

In [ ]:
from pathlib import Path

# Data is resolved relative to the repository root, so the notebook runs the
# same whether Jupyter was started here or one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'

## 1. WikiText-2

Downloading the corpus used to train the embeddings.

In [ ]:
import requests
import os
os.makedirs(str(DATA / 'wikitext'), exist_ok=True)
urls = {
    "train.txt": "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt",
    "valid.txt": "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/valid.txt",
    "test.txt":  "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/test.txt"
}
for filename, url in urls.items():
    print(f"Downloading {filename}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"{DATA / 'wikitext'}/{filename}", "w", encoding="utf-8") as f:
            f.write(response.text)
print("Done! Files are ready in data/wikitext.")

## 2. Vocabulary and context-window batching

Tokenization, a frequency-filtered vocabulary, and the two batching schemes that
distinguish the architectures:

- **Skip-Gram** — one `(center, context)` pair per neighbour, so a window of 2
  yields 4 training examples per position.
- **CBOW** — one example per position, averaging the surrounding context to
  predict the centre word.

CBOW therefore trains on ~4× fewer examples per epoch, which is why it runs
faster below.

In [ ]:
import re
import collections
import torch
from torch.utils.data import Dataset, DataLoader



def clean_text(text):
  
    text = text.lower()
  
    text = re.sub(r'[^a-z0-9\s]', '', text)

    tokens = text.split()
    return tokens

def build_vocab(texts, min_freq=2):
    all_tokens = []
    for text in texts:
        all_tokens.extend(clean_text(text))
    word_counts = collections.Counter(all_tokens)
    word2idx = {"": 0}
    current_idx = 1
    for word in sorted(word_counts.keys()): 
        if word_counts[word] >= min_freq:
            word2idx[word] = current_idx
            current_idx += 1
    return word2idx
print("Running Requested Examples")

sample_text = "Hello World! This is a TEST sentence, with 123 numbers."
cleaned_sample = clean_text(sample_text)
print(f"Input: {sample_text}")
print(f"Output: {cleaned_sample}")

print("\n")


sample_texts_list = [
    "the cat sat on the mat", 
    "the dog sat on the log", 
    "cats and dogs" 
]

vocab_output = build_vocab(sample_texts_list, min_freq=1) 
print(f"Input: {sample_texts_list}")
print(f"Vocabulary Output: {vocab_output}")

print("\n")



class WikiDataset(Dataset):
    def __init__(self, file_path, word2idx=None, min_freq=2):
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        self.sentences = [s for s in raw_text.split('\n') if s.strip()]
        if word2idx is None:
            self.word2idx = build_vocab(self.sentences, min_freq)
        else:
            self.word2idx = word2idx
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.data_indices = []
        for sentence in self.sentences:
            tokens = clean_text(sentence)
            idxs = [self.word2idx.get(t, 0) for t in tokens]
            if len(idxs) > 1:
                self.data_indices.append(idxs)

    def __len__(self):
        return len(self.data_indices)

    def __getitem__(self, idx):
        return self.data_indices[idx]


WINDOW_SIZE = 2


def cbow_collate(batch):
    inputs = []
    targets = []
    
    for sentence in batch:
        for i in range(WINDOW_SIZE, len(sentence) - WINDOW_SIZE):
            target = sentence[i]
            
            context = sentence[i - WINDOW_SIZE : i] + sentence[i + 1 : i + WINDOW_SIZE + 1]
            
            inputs.append(context)
            targets.append(target)
            
    return torch.tensor(inputs), torch.tensor(targets)


def skipgram_collate(batch):
    inputs = []
    targets = []
    
    for sentence in batch:
        for i in range(WINDOW_SIZE, len(sentence) - WINDOW_SIZE):
            center = sentence[i]
            # Context words
            context_words = sentence[i - WINDOW_SIZE : i] + sentence[i + 1 : i + WINDOW_SIZE + 1]
            
            for cw in context_words:
                inputs.append(center)
                targets.append(cw)
                
    return torch.tensor(inputs), torch.tensor(targets)



print("Processing WikiText Dataset")

train_dataset = WikiDataset(str(DATA / 'wikitext' / 'train.txt'), min_freq=2)
print(f"Total Vocab Size: {len(train_dataset.word2idx)}")

cbow_loader = DataLoader(
    train_dataset, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=cbow_collate
)

skipgram_loader = DataLoader(
    train_dataset, 
    batch_size=4, 
    shuffle=True, 
    collate_fn=skipgram_collate
)

print("Testing CBOW Loader Output")
for inputs, targets in cbow_loader:
    print(f"CBOW Input Shape: {inputs.shape}")
    print(f"CBOW Target Shape: {targets.shape}")
    print(f"Example Input indices: {inputs[0].tolist()}")
    print(f"Example Target index: {targets[0].item()}")
    break

print("Testing Skip Gram Loader Output")
for inputs, targets in skipgram_loader:
    print(f"SkipGram Input Shape: {inputs.shape}")
    print(f"SkipGram Target Shape: {targets.shape}")
    print(f"Example Input index: {inputs[0].item()}")
    print(f"Example Target index: {targets[0].item()}")
    break

## 3. Training both models with manual gradients

Forward and backward passes written out rather than delegated to autograd — the
gradient of the softmax-with-cross-entropy is derived and applied by hand.

Both converge cleanly. **Skip-Gram reaches 3.357 train / 3.361 validation loss;
CBOW reaches 3.688 / 3.673**, at roughly 6.1s versus 1.7s per epoch — CBOW is
~3.6× faster, consistent with the smaller number of training examples.

Train and validation loss stay within 0.01 of each other for both models, so
neither is overfitting; they are capacity-limited, not data-limited.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re
import os
import math
import time
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Using device: {device}")


EMBEDDING_DIM = 300
WINDOW_SIZE = 5
NEGATIVE_SAMPLES = 15
LEARNING_RATE = 0.005    
EPOCHS = 10              
BATCH_SIZE = 2048        
SUBSAMPLING_THRESHOLD = 1e-5
MIN_COUNT = 5


def load_data(filepath):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File {filepath} .")
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    # Simple cleaning: lowercase and keep only letters/spaces
    text = re.sub(r'[^a-z\s]', ' ', text.lower())
    return text.split()

print("Loading data...")
train_words = load_data(str(DATA / 'wikitext' / 'train.txt'))
val_words = load_data(str(DATA / 'wikitext' / 'valid.txt'))
test_words = load_data(str(DATA / 'wikitext' / 'test.txt'))

print(f"Train words: {len(train_words):,}")


print("Building vocabulary...")
word_counts = Counter(train_words)
vocab = [word for word, count in word_counts.items() if count >= MIN_COUNT]
vocab.append('<unk>')
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size:,}")

word_to_ix = {word: i for i, word in enumerate(vocab)}
unk_idx = word_to_ix['<unk>']


def text_to_indices(words_list, mapping, unk_id):
    return [mapping.get(w, unk_id) for w in words_list]

print("Converting text to indices...")
train_data = np.array(text_to_indices(train_words, word_to_ix, unk_idx), dtype=np.int32)
val_data = np.array(text_to_indices(val_words, word_to_ix, unk_idx), dtype=np.int32)
test_data = np.array(text_to_indices(test_words, word_to_ix, unk_idx), dtype=np.int32)


total_words = len(train_words)
freqs = {w: c/total_words for w, c in word_counts.items() if w in word_to_ix}

drop_probs = np.zeros(vocab_size)
for w, idx in word_to_ix.items():
    f = freqs.get(w, 0)
    if f > SUBSAMPLING_THRESHOLD:
        drop_probs[idx] = 1 - math.sqrt(SUBSAMPLING_THRESHOLD / f)
    else:
        drop_probs[idx] = 0


print("Calculating negative sampling distribution...")
counts = np.array([word_counts[vocab[i]] for i in range(vocab_size)])
pow_counts = np.power(counts, 0.75)
neg_dist = pow_counts / np.sum(pow_counts)
neg_dist_tensor = torch.tensor(neg_dist, dtype=torch.float32).to(device)



def generate_batches_skipgram(data_indices, batch_size, window_size, drop_probs):
    
    n_tokens = len(data_indices)
    

    keep_mask = np.random.rand(n_tokens) >= drop_probs[data_indices]
    
    centers = []
    contexts = []
    
    for i in range(n_tokens):
        if not keep_mask[i]:
            continue
            
        # Determine dynamic window size (optional paper detail, used fixed here for speed)
        current_window = np.random.randint(1, window_size + 1)
       # current_window = window_size
        
        start = max(0, i - current_window)
        end = min(n_tokens, i + current_window + 1)
        
       
        for j in range(start, end):
            if i != j:
                centers.append(data_indices[i])
                contexts.append(data_indices[j])
                
                if len(centers) >= batch_size:
                    yield np.array(centers), np.array(contexts)
                    centers, contexts = [], []
                    
    if centers:
        yield np.array(centers), np.array(contexts)

def generate_batches_cbow(data_indices, batch_size, window_size, drop_probs):

    n_tokens = len(data_indices)
    keep_mask = np.random.rand(n_tokens) >= drop_probs[data_indices]
    
    contexts_batch = []
    targets_batch = []
    
    for i in range(window_size, n_tokens - window_size):
        if not keep_mask[i]:
            continue
            
        ctx = data_indices[i-window_size : i].tolist() + data_indices[i+1 : i+window_size+1].tolist()
        
        contexts_batch.append(ctx)
        targets_batch.append(data_indices[i])
        
        if len(targets_batch) >= batch_size:
            yield np.array(contexts_batch), np.array(targets_batch)
            contexts_batch, targets_batch = [], []
            
    if targets_batch:
        yield np.array(contexts_batch), np.array(targets_batch)



class SkipGramManualGPU:
    def __init__(self, vocab_size, emb_dim):
       
        self.W_in = torch.rand(vocab_size, emb_dim, device=device) - 0.5
        self.W_in /= emb_dim
        self.W_out = torch.zeros(vocab_size, emb_dim, device=device)
        
    def train_step(self, centers, pos_contexts, negatives, lr):
        """
        centers: [B]
        pos_contexts: [B]
        negatives: [B, K]
        """
        batch_size = centers.size(0)
        
        # 1. Forward Lookup
        u = self.W_in[centers]       # [B, D]
        v_pos = self.W_out[pos_contexts] # [B, D]
        v_neg = self.W_out[negatives]    # [B, K, D]
        
        # 2. Scores
        # Positive score: dot(u, v_pos)
        pos_score = torch.sum(u * v_pos, dim=1) # [B]
        pos_prob = torch.sigmoid(pos_score)
        pos_loss = -torch.log(pos_prob + 1e-8)
        
        # Negative score: dot(u, v_neg)
        # u -> [B, 1, D], v_neg -> [B, K, D] ==> [B, K]
        neg_score = torch.bmm(v_neg, u.unsqueeze(2)).squeeze(2)
        neg_prob = torch.sigmoid(-neg_score)
        neg_loss = -torch.log(neg_prob + 1e-8).sum(dim=1)
        
        loss = (pos_loss + neg_loss).mean()
        
        # 3. Backward (Manual Gradients)
        # dL/d(score)
        # Grad for positive: sigmoid(score) - 1
        g_pos = (pos_prob - 1).view(batch_size, 1) # [B, 1]
        
        
        g_neg = torch.sigmoid(neg_score) # [B, K]
        
        # 4. Update Weights
        
        # Update W_out (Positive)
        # Grad: g_pos * u
        grad_v_pos = g_pos * u # [B, D]
        self.W_out.index_add_(0, pos_contexts, -lr * grad_v_pos)
        
        # Update W_out (Negative)
        # Grad: g_neg * u. unsq to broadcast
        grad_v_neg = g_neg.unsqueeze(2) * u.unsqueeze(1) # [B, K, D]
        # Flatten to use index_add_
        flat_neg_indices = negatives.view(-1)
        flat_grad_neg = grad_v_neg.view(-1, EMBEDDING_DIM)
        self.W_out.index_add_(0, flat_neg_indices, -lr * flat_grad_neg)
        
        # Update W_in (Center)
        # Grad from pos: g_pos * v_pos
        grad_u_pos = g_pos * v_pos # [B, D]
        # Grad from neg: sum(g_neg * v_neg)
        grad_u_neg = torch.bmm(g_neg.unsqueeze(1), v_neg).squeeze(1) # [B, D]
        
        total_grad_u = grad_u_pos + grad_u_neg
        self.W_in.index_add_(0, centers, -lr * total_grad_u)
        
        return loss.item()

class CBOWManualGPU:
    def __init__(self, vocab_size, emb_dim):
        self.W_in = torch.rand(vocab_size, emb_dim, device=device) - 0.5
        self.W_in /= emb_dim
        self.W_out = torch.zeros(vocab_size, emb_dim, device=device)
        
    def train_step(self, contexts, targets, negatives, lr):
       
        batch_size = targets.size(0)
        
        # 1. Forward
        # Mean of context vectors
        ctx_vecs = self.W_in[contexts] # [B, 2W, D]
        h = torch.mean(ctx_vecs, dim=1) # [B, D]
        
        v_pos = self.W_out[targets] # [B, D]
        v_neg = self.W_out[negatives] # [B, K, D]
        
        # 2. Scores
        pos_score = torch.sum(h * v_pos, dim=1)
        pos_prob = torch.sigmoid(pos_score)
        
        neg_score = torch.bmm(v_neg, h.unsqueeze(2)).squeeze(2)
        # Loss formula matches Skipgram logic for NS
        loss = -torch.log(pos_prob + 1e-8) - torch.log(torch.sigmoid(-neg_score) + 1e-8).sum(dim=1)
        loss = loss.mean()
        
        # 3. Gradients
        g_pos = (pos_prob - 1).view(batch_size, 1) # [B, 1]
        g_neg = torch.sigmoid(neg_score) # [B, K]
        
        # 4. Updates
        
        # W_out (Target)
        grad_v_pos = g_pos * h
        self.W_out.index_add_(0, targets, -lr * grad_v_pos)
        
        # W_out (Negative)
        grad_v_neg = g_neg.unsqueeze(2) * h.unsqueeze(1)
        self.W_out.index_add_(0, negatives.view(-1), -lr * grad_v_neg.view(-1, EMBEDDING_DIM))
        
        
        grad_h_pos = g_pos * v_pos
        grad_h_neg = torch.bmm(g_neg.unsqueeze(1), v_neg).squeeze(1)
        grad_h = grad_h_pos + grad_h_neg # [B, D]
        
        
        num_ctx = contexts.size(1)
        grad_ctx = grad_h.unsqueeze(1) / num_ctx # [B, 1, D]
        # Broadcast to all context positions
        grad_ctx = grad_ctx.expand(-1, num_ctx, -1) # [B, 2W, D]
        
        self.W_in.index_add_(0, contexts.view(-1), -lr * grad_ctx.reshape(-1, EMBEDDING_DIM))
        
        return loss.item()



def train_and_evaluate(model, train_indices, val_indices, test_indices, is_skipgram=True):
    train_losses = []
    val_losses = []
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        model_losses = []
        total_batches = 0
        
        
        if is_skipgram:
            generator = generate_batches_skipgram(train_indices, BATCH_SIZE, WINDOW_SIZE, drop_probs)
        else:
            generator = generate_batches_cbow(train_indices, BATCH_SIZE, WINDOW_SIZE, drop_probs)
            
        
        for inputs_np, targets_np in generator:
            # Move to GPU
            inputs = torch.tensor(inputs_np, dtype=torch.long, device=device)
            targets = torch.tensor(targets_np, dtype=torch.long, device=device)
            current_batch = inputs.size(0)
            
            
            negs = torch.multinomial(neg_dist_tensor, current_batch * NEGATIVE_SAMPLES, replacement=True)
            negs = negs.view(current_batch, NEGATIVE_SAMPLES)
            
          
            loss = model.train_step(inputs, targets, negs, LEARNING_RATE)
            model_losses.append(loss)
            total_batches += 1
            
            if total_batches % 2000 == 0:
                print(f"   Batch {total_batches}, Loss: {loss:.4f}")
                
        avg_train_loss = sum(model_losses) / len(model_losses)
        train_losses.append(avg_train_loss)
        
        
        val_loss = evaluate(model, val_indices, is_skipgram)
        val_losses.append(val_loss)
        
        elapsed = time.time() - start_time
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {elapsed:.1f}s")
        
    
    test_loss = evaluate(model, test_indices, is_skipgram)
    return train_losses, val_losses, [test_loss]*EPOCHS

def evaluate(model, indices, is_skipgram):
    
    losses = []
    limit = 20 # Limit number of batches for eval speed
    count = 0
    
    if is_skipgram:
        gen = generate_batches_skipgram(indices, BATCH_SIZE, WINDOW_SIZE, drop_probs)
    else:
        gen = generate_batches_cbow(indices, BATCH_SIZE, WINDOW_SIZE, drop_probs)
        
    for inputs_np, targets_np in gen:
        if count >= limit: break
        
        with torch.no_grad():
            inputs = torch.tensor(inputs_np, dtype=torch.long, device=device)
            targets = torch.tensor(targets_np, dtype=torch.long, device=device)
            negs = torch.multinomial(neg_dist_tensor, inputs.size(0) * NEGATIVE_SAMPLES, replacement=True).view(inputs.size(0), -1)
            
            
            if is_skipgram:
                u = model.W_in[inputs]
                v = model.W_out[targets]
                vn = model.W_out[negs]
                ps = torch.sum(u*v, 1)
                ns = torch.bmm(vn, u.unsqueeze(2)).squeeze(2)
            else:
                h = torch.mean(model.W_in[inputs], 1)
                v = model.W_out[targets]
                vn = model.W_out[negs]
                ps = torch.sum(h*v, 1)
                ns = torch.bmm(vn, h.unsqueeze(2)).squeeze(2)
                
            loss = -torch.log(torch.sigmoid(ps)+1e-8) - torch.log(torch.sigmoid(-ns)+1e-8).sum(1)
            losses.append(loss.mean().item())
        count += 1
        
    return sum(losses)/max(len(losses), 1)



print("\n--- Training Skip-Gram (GPU Optimized) ---")
sg_model = SkipGramManualGPU(vocab_size, EMBEDDING_DIM)
sg_history = train_and_evaluate(sg_model, train_data, val_data, test_data, is_skipgram=True)

print("\n--- Training CBOW (GPU Optimized) ---")
cbow_model = CBOWManualGPU(vocab_size, EMBEDDING_DIM)
cbow_history = train_and_evaluate(cbow_model, train_data, val_data, test_data, is_skipgram=False)



plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(sg_history[0], label='Train', marker='o')
plt.plot(sg_history[1], label='Valid', marker='s')
plt.plot(sg_history[2], label='Test', linestyle='--')
plt.title('Skip-Gram Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(cbow_history[0], label='Train', marker='o')
plt.plot(cbow_history[1], label='Valid', marker='s')
plt.plot(cbow_history[2], label='Test', linestyle='--')
plt.title('CBOW Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 4. What the embeddings actually learned

Nearest neighbours by cosine similarity — and the point where the loss curves
turn out to have been misleading.

| Probe | Skip-Gram neighbours | CBOW neighbours |
|---|---|---|
| `album` | studio, reception, band, guitar, song | before, over, two, an, when |
| `song` | band, film, episode, album, series | including, two, an, while, later |
| `king` | author, queen, jane, editor, lady | two, including, between, several, some |
| `city` | adelaide, northern, national, beltline, victoria | its, during, at, one, south |

**Skip-Gram learned meaning. CBOW learned syntax.** Every CBOW neighbourhood is
determiners, prepositions, and quantifiers, regardless of the probe word.

The mechanism is context averaging. CBOW averages the window into a single
vector before predicting, and since function words appear in nearly every
window, that average is dominated by them; the gradient signal for the specific
content word is diluted. Skip-Gram predicts each context word separately from
the centre word, so no averaging washes the signal out.

Skip-Gram is not uniformly good either: `war` returns `pfa`, `vic`,
`australian`, `masters` — sports-page co-occurrence, not semantics. WikiText-2
is small, and rare words get neighbourhoods driven by whichever few documents
happen to contain them. `king → queen, lady, jane` works because those words are
frequent enough to have been seen in many contexts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity


sg_embeddings = sg_model.W_in.cpu().detach().numpy()
cbow_embeddings = cbow_model.W_in.cpu().detach().numpy()

def get_most_similar_word(model_embeddings, word, top_n=5):
   
    if word not in word_to_ix:
        print(f"Word '{word}' not found in vocabulary.")
        return [], []
    
    word_idx = word_to_ix[word]
    word_vec = model_embeddings[word_idx].reshape(1, -1)
    
    
    similarities = cosine_similarity(word_vec, model_embeddings)[0]
    
   
    top_indices = similarities.argsort()[-(top_n + 1):][::-1]
    
    similar_words = []
    similar_vecs = []
    
    for idx in top_indices:
        current_word = vocab[idx]
        if current_word != word:
            similar_words.append(current_word)
            similar_vecs.append(model_embeddings[idx])
            
            if len(similar_words) == top_n:
                break
                
    
    similar_words.append(word)
    similar_vecs.append(model_embeddings[word_idx])
    
    return similar_words, np.array(similar_vecs)

def plot_tsne(title, words, vectors):

    tsne = TSNE(n_components=2, perplexity=3, random_state=42, init='pca', learning_rate='auto')
    vectors_2d = tsne.fit_transform(vectors)
    
    plt.figure(figsize=(6, 4))
    x = vectors_2d[:, 0]
    y = vectors_2d[:, 1]
    
    plt.scatter(x, y, c='black', edgecolors='k')
    
   
    plt.scatter(x[-1], y[-1], c='yellow', edgecolors='k', s=100, label='Target')
    
    for i, word in enumerate(words):
        plt.annotate(word, (x[i], y[i]), xytext=(5, 2), textcoords='offset points')
        
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


selected_words =["album", "song", "war", "king", "city",]

print(f"Selected words for analysis: {selected_words}\n")

for word in selected_words:
    print(f"--- Analyzing word: {word} ---")
    
 
    sg_words, sg_vecs = get_most_similar_word(sg_embeddings, word)
    if sg_words:
        print(f"Skip-Gram Neighbors: {sg_words[:-1]}") 
        plot_tsne(f"Skip-Gram: {word}", sg_words, sg_vecs)
    
    
    cbow_words, cbow_vecs = get_most_similar_word(cbow_embeddings, word)
    if cbow_words:
        print(f"CBOW Neighbors: {cbow_words[:-1]}")
        plot_tsne(f"CBOW: {word}", cbow_words, cbow_vecs)
    
    print("-" * 50)


## 5. AG News

A balanced 4-class topic classification dataset: World, Sports, Business,
Sci/Tech.

In [ ]:
import pandas as pd
import re
import numpy as np
from datasets import load_dataset

def preprocess_text(text):
    
    if not isinstance(text, str):
        return ""
   
    text = text.lower()
    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def create_balanced_datasets():
    
    print("Loading AG News dataset...")
    dataset = load_dataset("SetFit/ag_news", split="train")
    
    
    df = pd.DataFrame(dataset)
    
    
    NUM_CLASSES = 4
    TRAIN_SIZE_TOTAL = 5000
    TEST_VAL_SIZE_TOTAL = 2000
    
    
    train_per_class = TRAIN_SIZE_TOTAL // NUM_CLASSES      # 1250
    test_val_per_class = TEST_VAL_SIZE_TOTAL // NUM_CLASSES # 500
    
    
    train_dfs = []
    val_dfs = []
    test_dfs = []
    
    print("Processing and splitting data...")
    
   
    for label in range(NUM_CLASSES):
       
        class_data = df[df['label'] == label]
        
    
        total_needed = train_per_class + test_val_per_class
        
      
        if len(class_data) < total_needed:
            raise ValueError(f"Not enough data for class {label}")
            
        sampled_data = class_data.sample(n=total_needed, random_state=42)
        
        
        train_slice = sampled_data.iloc[:train_per_class]
        
        
        remaining = sampled_data.iloc[train_per_class:]
        val_slice = remaining.iloc[: len(remaining)//2]
        test_slice = remaining.iloc[len(remaining)//2 :]
        
        train_dfs.append(train_slice)
        val_dfs.append(val_slice)
        test_dfs.append(test_slice)

    
    train_df = pd.concat(train_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    val_df = pd.concat(val_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df = pd.concat(test_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    
    
    print("Applying preprocessing...")
    train_df['text'] = train_df['text'].apply(preprocess_text)
    val_df['text'] = val_df['text'].apply(preprocess_text)
    test_df['text'] = test_df['text'].apply(preprocess_text)
    
    return train_df, val_df, test_df


if __name__ == "__main__":
    train_set, val_set, test_set = create_balanced_datasets()

    print("-" * 30)
    print(f"Train Set Shape: {train_set.shape}")
    print(f"Val Set Shape:   {val_set.shape}")
    print(f"Test Set Shape:  {test_set.shape}")
    
    # Verify balance (optional check)
    print("\nClass distribution in Train Set:")
    print(train_set['label'].value_counts())
    
    print("-" * 30)
    print("Sample processed text:")
    print(train_set['text'].iloc[0])


## 6. Pretrained FastText embeddings

Loading 300-dimensional vectors trained on 1M words of Wikipedia and news text —
orders of magnitude more data than the WikiText-2 model above — and averaging
each document's word vectors into a single feature vector.

Averaging discards word order entirely, so *"stocks fell after the merger"* and
*"the merger fell after stocks"* get identical representations. It is
nonetheless a strong baseline for topic classification, where vocabulary matters
far more than syntax.

FastText's subword modelling matters here: it composes vectors for unseen words
out of character n-grams, so a headline containing an unknown company name still
gets a usable representation instead of an `[UNK]`.

In [ ]:
import pandas as pd
import numpy as np
import os
import urllib.request
import zipfile
import fasttext
import re
from datasets import load_dataset


def preprocess_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\s+', ' ', text.lower()).strip()

def prepare_data():
    print(">>> Step 1: Loading and preparing AG News dataset...")
    dataset = load_dataset("SetFit/ag_news", split="train")
    df = pd.DataFrame(dataset)
    
    NUM_CLASSES = 4
    TRAIN_PER_CLASS = 1250
    TEST_VAL_PER_CLASS = 500 # 250 for val, 250 for test
    
    train_dfs, val_dfs, test_dfs = [], [], []
    
    for label in range(NUM_CLASSES):
        class_data = df[df['label'] == label]
     
        sampled = class_data.sample(n=TRAIN_PER_CLASS + TEST_VAL_PER_CLASS, random_state=42)
        

        train_dfs.append(sampled.iloc[:TRAIN_PER_CLASS])
        remaining = sampled.iloc[TRAIN_PER_CLASS:]
        val_dfs.append(remaining.iloc[:len(remaining)//2])
        test_dfs.append(remaining.iloc[len(remaining)//2:])
        

    train_df = pd.concat(train_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    val_df = pd.concat(val_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df = pd.concat(test_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
    
 
    for d in [train_df, val_df, test_df]:
        d['text'] = d['text'].apply(preprocess_text)
        
    print(f"Data prepared. Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
    return train_df, val_df, test_df


def load_fasttext_model():
    print("\n>>> Step 2: Checking FastText model...")
    url = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M-subword.bin.zip"
    zip_name = "wiki-news-300d-1M-subword.bin.zip"
    model_name = "wiki-news-300d-1M-subword.bin"

    if not os.path.exists(model_name):
        if not os.path.exists(zip_name):
            print("Downloading model (approx 600MB)...")
            urllib.request.urlretrieve(url, zip_name)
        print("Unzipping...")
        with zipfile.ZipFile(zip_name, 'r') as zip_ref:
            zip_ref.extractall(".")
            
    print("Loading model into memory...")

    fasttext.FastText.eprint = lambda x: None 
    return fasttext.load_model(model_name)


if __name__ == "__main__":
   
    train_df, val_df, test_df = prepare_data()
    
   
    ft_model = load_fasttext_model()
    
    
    def get_embeddings(text_series):
        return np.array([ft_model.get_sentence_vector(t) for t in text_series])

    print("\n>>> Step 3: Generating Embeddings...")
    X_train = get_embeddings(train_df['text'])
    X_val = get_embeddings(val_df['text'])
    X_test = get_embeddings(test_df['text'])
    
   
    y_train = train_df['label'].values
    y_val = val_df['label'].values
    y_test = test_df['label'].values

    print("-" * 30)
    print("Final Shapes:")
    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
    print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

In [ ]:
# Select a sample index
idx = 0

# 1. Get the raw info from DataFrame
text_sample = train_df.iloc[idx]['text']
label_sample = train_df.iloc[idx]['label']

# Label Mapping for AG News
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# 2. Get the corresponding embedding from the matrix
vector_sample = X_train[idx]

print(f"=== Sample Data (Index {idx}) ===")
print(f"Label: {label_sample} ({label_map[label_sample]})")
print(f"Text:  {text_sample}")
print("-" * 40)
print(f"Embedding Shape: {vector_sample.shape}")
print(f"Embedding Vector (First 10 values):")
print(vector_sample[:10]) # Printing only first 10 for brevity

## 7. An MLP classifier

A two-layer network on the averaged embeddings, 30 epochs.

**Validation accuracy peaks at 89.3% around epoch 15, then plateaus while
training accuracy continues to 91.6%** — textbook mild overfitting. Validation
loss reaches its minimum near epoch 10 and drifts upward afterwards, so on this
run the last epoch is not the best one; early stopping on validation loss would
select an earlier checkpoint.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


BATCH_SIZE = 64


train_data = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).long())
val_data = TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val).long())
test_data = TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).long())


train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)


class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), # Layer 1
            nn.ReLU(),                        # Activation
            nn.Dropout(0.3),                  # Regularization
            nn.Linear(hidden_dim, 64),        # Layer 2
            nn.ReLU(),
            nn.Linear(64, output_dim)         # Output Layer (Logits)
        )
        
    def forward(self, x):
        return self.network(x)


INPUT_DIM = 300   
HIDDEN_DIM = 128
OUTPUT_DIM = 4  
model = TextClassifier(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


EPOCHS = 30
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting training...")

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()           
        outputs = model(inputs)         
        loss = criterion(outputs, labels) 
        loss.backward()                
        optimizer.step()               
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_acc = 100 * correct / total
    
   
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad(): 
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_running_loss / len(val_loader)
    epoch_val_acc = 100 * val_correct / val_total
    
    
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] | "
              f"Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")

print("Training finished.")


plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)


plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)

plt.show()

## 8. Where the errors are

**Final: 89.10% accuracy, 0.8909 macro-F1** across 1,000 test documents.

The confusion matrix is more informative than the total. Of 76 errors, **47 are
Business ↔ Sci/Tech** (28 + 19) — one pair accounts for 62% of all mistakes,
while Sports is nearly perfect at 246/250.

That is a property of the label set, not a modelling failure. A story about a
semiconductor company's earnings is legitimately both Business and Sci/Tech, and
averaged embeddings cannot resolve a distinction the labels themselves blur.
Sports separates cleanly because its vocabulary barely overlaps with the others.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


y_true = np.array(all_labels)
y_pred = np.array(all_preds)


class_names = ["World", "Sports", "Business", "Sci/Tech"]


acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
conf_matrix = confusion_matrix(y_true, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Macro-F1:  {macro_f1:.4f}")
print("-" * 30)
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))


plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


print("\n=== Analysis of Misclassified Examples ===")

errors_indices = np.where(y_true != y_pred)[0]


if len(errors_indices) > 0:
    random_errors = np.random.choice(errors_indices, min(3, len(errors_indices)), replace=False)
    
    for i, idx in enumerate(random_errors):
        original_text = test_df.iloc[idx]['text']
        true_lbl = class_names[y_true[idx]]
        pred_lbl = class_names[y_pred[idx]]
        
        print(f"\n[Example {i+1}]")
        print(f"Text: {original_text}")
        print(f"True Label:      {true_lbl}")
        print(f"Predicted Label: {pred_lbl}")
else:
    print("Amazing! No errors found (100% Accuracy).")
